In [14]:
from importlib import reload ,import_module
import module.utilize as utilize
import module.multiVariant as multiVariant
import module.singleVariant as singleVariant
import module.multiHistogramBase as multiHistogramBase
import numpy as np
from numba import njit,jit, float32
import module.singleVariantCopulaBase as CopulaBase
from tqdm import tqdm
import time
from multiprocessing import Pool
from sklearn.metrics import root_mean_squared_error
import cupy as cp
import module.multiHistogramSparse as multiHistogramSparse
import module.ananlysisFuncion as ananlysisFunction
reload(utilize)
reload(multiVariant)
reload(singleVariant)
reload(multiHistogramBase)
reload(CopulaBase)
reload(multiHistogramSparse)
reload(ananlysisFunction)

startTime=time.time()
attribute_names=np.array(["phi_grav","particle_mass_density","zmom","ymom","xmom"])
#attribute_names=np.array(["phi_grav","particle_mass_density"])
incremental_number=300
all_ensamble_data=np.empty([0,incremental_number,64,64,64])

for name in attribute_names:
    data=utilize.readFiles(name,incremental_number)
    data=data.reshape(1,incremental_number,64,64,64)
    all_ensamble_data=np.append(all_ensamble_data,data,axis=0)

#print(all_ensamble_data.shape)
#print(all_ensamble_data[0].shape)
covBlockSize=5
dataBlockSize=6
binsNumber=128
sizeZ=60
sizeY=60
sizeX=60
minMaxBlockSize=2
isMinMax=False

for i in range(2,6):
    data=all_ensamble_data[0:i, :, :, :, :]
    print("start fit model")
    with tqdm(total=2, desc="Model fitting") as pbar:
        #oursModel=multiVariant.multiDistCopula3D(all_ensamble_data,dataBlockSize,covBlockSize,binsNumber,[sizeZ,sizeY,sizeX],minMaxBlockSize,isMinMax)
        oursModel=multiVariant.multiDistCopula3D.load(f"Nyx_{i}varaibles_{incremental_number}members_{binsNumber}Bins_dBlock{dataBlockSize}_cBlock5")
        #conditions=np.array([[0,1e5],[3e10,5e10]])
        #oursModel.fit()
        print("ours complete fit")
        pbar.update(1)
        
        gtModel=multiHistogramSparse.multiHistogramSpaseModel(data,blockSize=1,binsNumber=binsNumber)
        gtModel.fit()

        multiBinEdges=gtModel.vBinEdges

        print("complete fit")
        pbar.update(1)

    oursError=[]



    with tqdm(total=sizeZ*sizeY*sizeX, desc="總進度") as pbar:
        for idx in range(sizeZ * sizeY * sizeX):
            
            z = idx // (sizeY * sizeX)
            y = (idx // sizeX) % sizeY
            x = idx % sizeX        
            ### GroundTruth ###

            gtMultiHistModel=gtModel.getHistByPos(z,y,x)
            coords_gt, vals_gt = gtMultiHistModel.to_coo()
            ### ours method ###

            oursSamples=oursModel.sampleByPos(z,y,x)
            oursHistModel=multiHistogramSparse.SparseMultiHistogramBlock(bin_edges=multiBinEdges)
            oursHistModel.add_samples(oursSamples)
            oursHistModel.normalize()
            
            coords_target,vals_target=oursHistModel.to_coo()
            emd=ananlysisFunction.emd_sparse(coords_gt,vals_gt,coords_target,vals_target)
            oursError.append(emd)
            
            pbar.update(1)


    oursError=np.array(oursError)
    oursError=oursError.mean()

    with open(f"NyxSelfEMD_DBlock{dataBlockSize}_Bin{binsNumber}.txt", "a", encoding="utf-8") as f:  # 使用 "a" 表示 append
        f.write(f"Variable:{i} ,binNumber:{binsNumber}, oursError:{oursError}\n")  # 每次寫入並換行

start fit model


Model fitting:   0%|          | 0/2 [00:00<?, ?it/s]

ours complete fit


Model fitting: 100%|██████████| 2/2 [01:00<00:00, 30.15s/it]


complete fit


...

Model fitting:  50%|█████     | 1/2 [00:00<00:00,  7.03it/s]

ours complete fit


Model fitting: 100%|██████████| 2/2 [03:03<00:00, 91.58s/it] 


complete fit


總進度: 100%|██████████| 216000/216000 [58:58<00:00, 61.04it/s] 


In [15]:
oursModel=multiVariant.multiDistCopula3D.load(f"Nyx_2varaibles_{incremental_number}members_128Bins_dBlock6_cBlock5")
oursModel.singleDistModels[0].blocks.__len__()

1000